In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
import pickle

# 1. Load your raw (unaligned) data
# Replace with your actual raw pkl path
with open('dataset/bci/raw/physionet_mi_raw.pkl', 'rb') as f:
    data = pickle.load(f)

# 2. Prepare features for t-SNE
# We flatten the trials (Channels x Time) into a single vector per trial
X = data['x']  # Shape: (Trials, Channels, Time)
y_subject = data['subject']
y_label = data['y']

X_flat = X.reshape(X.shape[0], -1)

# 3. Run t-SNE
print("Computing t-SNE embedding... (this may take a minute)")
tsne = TSNE(n_components=2, perplexity=30, random_state=42)
X_embedded = tsne.fit_transform(X_flat)

# 4. Plot: Color by Subject ID (Showing the "Distinct Clusters" problem)
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
sns.scatterplot(x=X_embedded[:,0], y=X_embedded[:,1], hue=y_subject, palette='viridis', legend=None)
plt.title("Raw EEG: Colored by Subject ID\n(Showing Domain Shift/Distinct Clusters)")
plt.xlabel("t-SNE 1")
plt.ylabel("t-SNE 2")

# 5. Plot: Color by Label (Showing why it's hard to classify raw data)
plt.subplot(1, 2, 2)
sns.scatterplot(x=X_embedded[:,0], y=X_embedded[:,1], hue=y_label, palette='coolwarm')
plt.title("Raw EEG: Colored by Intent (Left/Right)\n(No clear separation yet)")
plt.xlabel("t-SNE 1")
plt.ylabel("t-SNE 2")

plt.tight_layout()
plt.show()

In [ ]:
import mne
import matplotlib.pyplot as plt
import numpy as np

# 1. Visualize the 10-10 System (Biological Spatial Topography)
# This is what Depthwise Convolutions learn!
info = mne.create_info(ch_names=data['ch_names'], sfreq=160, ch_types='eeg')
info.set_montage('standard_1010')

fig, ax = plt.subplots(1, 2, figsize=(12, 5))

# Plot the sensor locations
info.plot_sensors(show_names=True, axes=ax[0])
ax[0].set_title("10-10 System: Biological Spatial Map\n(Target for Depthwise Conv)")

# 2. Visualize the Power Spectral Density (Brain Rhythms)
# This is what Temporal Convolutions learn!
# We pick a central motor electrode (C3)
c3_idx = data['ch_names'].index('C3')
psds, freqs = mne.time_frequency.psd_array_multitaper(data['x'][:, c3_idx, :], sfreq=160, fmin=2, fmax=40)

# Average across trials
avg_psd = 10 * np.log10(psds.mean(0))
ax[1].plot(freqs, avg_psd, color='darkblue', lw=2)
ax[1].fill_between(freqs, avg_psd, alpha=0.3, color='cyan')
ax[1].axvspan(8, 12, color='red', alpha=0.1, label='$\mu$ band')
ax[1].axvspan(13, 30, color='green', alpha=0.1, label='$\beta$ band')

ax[1].set_title("Frequency Spectrum: Brain Rhythms\n(Target for Temporal Conv)")
ax[1].set_xlabel("Frequency (Hz)")
ax[1].set_ylabel("Power (dB)")
ax[1].legend()

plt.tight_layout()
plt.show()